# 3 — Modeling & Evaluation
Notebook ini memuat `data/processed/dataset_clean.csv`, split train/test, scaling, training model, evaluasi, dan menyimpan model terbaik.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import DataPreprocessor
from src.model import ModelBuilder
from src.evaluation import ModelEvaluator

PROCESSED_PATH = ROOT / 'data' / 'processed' / 'dataset_clean.csv'
MODEL_OUT = ROOT / 'results' / 'models' / 'best_model.pkl'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH

WindowsPath('C:/Users/bobok/modul2_datamining/data/processed/dataset_clean.csv')

In [2]:
# Load processed data
if not PROCESSED_PATH.exists():
    raise FileNotFoundError('dataset_clean.csv belum ada. Jalankan notebook 2_preprocessing dulu.')

df = pd.read_csv(PROCESSED_PATH)
print('Shape:', df.shape)
df.head()

Shape: (54, 6)


,age,income,education,experience,credit_score,purchased
0,25,30000,1,1,650,0
1,30,35000,0,3,680,0
2,35,40000,0,5,700,1
3,40,45000,2,7,720,1
4,45,50000,2,9,750,1


In [3]:
# Prepare features & target
preferred_targets = ['purchased', 'loan_approved', 'target', 'label']
target_col = next((c for c in preferred_targets if c in df.columns), None)
if target_col is None:
    target_col = df.columns[-1]
    print('⚠️ Target column tidak ditemukan di daftar default; memakai kolom terakhir:', target_col)
else:
    print('✓ Target column:', target_col)

X = df.drop(columns=[target_col])
y = df[target_col]

X.shape, y.shape

✓ Target column: purchased


((54, 5), (54,))

In [4]:
# Split + scaling (fit on train, transform on test)
builder = ModelBuilder()
X_train, X_test, y_train, y_test = builder.split_data(X, y, test_size=0.2)

scaler = DataPreprocessor()
X_train_s = scaler.normalize_scale(X_train, X_train.columns, method='standard', fit=True)
X_test_s = scaler.normalize_scale(X_test, X_test.columns, method='standard', fit=False)

X_train_s.head()

✓ Data split selesai
  Train set: 43 samples
  Test set: 11 samples
✓ Data di-scale menggunakan Standard Scaler
✓ Data di-scale menggunakan Standard Scaler


,age,income,education,experience,credit_score
8,-0.222393,-0.328134,-1.481872,-0.239535,-0.310602
26,-0.616739,-0.515018,-1.481872,-0.484773,-0.409681
6,-1.208258,-1.262554,-0.501557,-1.220488,-1.301394
34,0.763472,0.606287,0.478759,0.741418,0.640558
4,0.467713,0.419403,0.478759,0.496180,0.482032


In [5]:
# Train models
model_lr = builder.train_logistic_regression(X_train_s, y_train)
model_rf = builder.train_random_forest(X_train_s, y_train, n_estimators=200)

✓ Logistic Regression model dilatih
✓ Random Forest model dilatih (200 trees)


In [6]:
# Evaluate
evaluator = ModelEvaluator()

y_pred_lr = model_lr.predict(X_test_s)
y_pred_rf = model_rf.predict(X_test_s)

print('Logistic Regression:')
m_lr = evaluator.evaluate_classification(y_test, y_pred_lr)

print('Random Forest:')
m_rf = evaluator.evaluate_classification(y_test, y_pred_rf)

comparison = evaluator.compare_models({'LogReg': model_lr, 'RandomForest': model_rf}, X_test_s, y_test)
comparison

Logistic Regression:
CLASSIFICATION METRICS
accuracy       : 0.9091
precision      : 0.9192
recall         : 0.9091
f1             : 0.9027
Random Forest:
CLASSIFICATION METRICS
accuracy       : 0.9091
precision      : 0.9192
recall         : 0.9091
f1             : 0.9027

MODEL COMPARISON
       Model  Accuracy
      LogReg  0.909091
RandomForest  0.909091


,Model,Accuracy
0,LogReg,0.909091
1,RandomForest,0.909091


In [7]:
# Save best model
best_model = model_rf if m_rf.get('accuracy', 0) >= m_lr.get('accuracy', 0) else model_lr
builder.save_model(best_model, str(MODEL_OUT))
MODEL_OUT

✓ Model disimpan ke: C:\Users\bobok\modul2_datamining\results\models\best_model.pkl


WindowsPath('C:/Users/bobok/modul2_datamining/results/models/best_model.pkl')

## Selesai
- Model terbaik tersimpan di `results/models/best_model.pkl`
- Anda bisa lanjut melakukan tuning dengan `ModelBuilder.hyperparameter_tuning()` jika diperlukan.